### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We now use the [Alpaca dataset](https://huggingface.co/datasets/unsloth/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read our docs [here](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Conversational.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for tokens, tags in zip(examples["tokens"], examples["ner_tags"]):
        if tokens[0] == "RT":
            tokens = tokens[3:]
            tags = tags[3:]
        input = " ".join(tokens)
        output = ""
        in_mention = False
        for token, tag_int in zip(tokens, tags):
            tag = tag_int2str(tag_int)
            if tag == "O":
                if in_mention:
                    output += f">> {token} "
                    in_mention = False
                else:
                    output += f"{token} "
            elif tag.startswith("B"):
                if in_mention:
                    output += f">> <<{tag[2:]}: {token} "
                else:
                    output += f"<<{tag[2:]}: {token} "
                    in_mention = True
            else:
                output += f"{token} "
        if in_mention:
            output += ">>"
        texts.append(prompt.format(input, output.rstrip()) + tokenizer.eos_token)
    return { "text" : texts, }

In [ ]:
prompt = """You are an expert annotator.
Find named entities in the input and enclose them with << and >>.
Add the entity type (PER:, LOC:, ORG:, or MISC:) after each <<.
Return only the input sentence with the added labels.

### Input:
{}

### Response:
{}"""

from datasets import load_dataset
train_data = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = "train")
tag_int2str = train_data.features["ner_tags"].feature.int2str
train_data = train_data.map(formatting_prompts_func, batched = True)

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

for split in ["dev", "test"]:
    eval_data = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = split)
    with open(f"llama-{split}-pred.txt", "w", encoding="utf-8") as file:
        for row in eval_data:
            tokens = row["tokens"][3:] if row["tokens"][0] == "RT" else row["tokens"]
            input = tokenizer(
            [
                prompt.format(
                    " ".join(tokens), # input
                    "", # output - leave this blank for generation!
                )
            ], return_tensors = "pt").to("cuda")
            output = model.generate(**input, use_cache = True)
            output = tokenizer.batch_decode(output[:, input.input_ids.shape[1]:], skip_special_tokens = True)
            print(output)
            file.write("\n".join(output) + "\n")

In [ ]:
def convert_to_inline_format(examples):
    texts = []
    for tokens, tags in zip(examples["tokens"], examples["ner_tags"]):
        if tokens[0] == "RT":
            tokens = tokens[3:]
            tags = tags[3:]
        output = ""
        in_mention = False
        for token, tag_int in zip(tokens, tags):
            tag = tag_int2str(tag_int)
            if tag == "O":
                if in_mention:
                    output += f">> {token} "
                    in_mention = False
                else:
                    output += f"{token} "
            elif tag.startswith("B"):
                if in_mention:
                    output += f">> <<{tag[2:]}: {token} "
                else:
                    output += f"<<{tag[2:]}: {token} "
                    in_mention = True
            else:
                output += f"{token} "
        if in_mention:
            output += ">>"
        texts.append(output.rstrip())
    return { "text" : texts, }

for split in ["dev", "test"]:
    eval_data = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = split)
    tag_int2str = eval_data.features["ner_tags"].feature.int2str
    eval_data = eval_data.map(convert_to_inline_format, batched = True,)
    with open(f"llama-{split}-true.txt", "w", encoding="utf-8") as file:
        for sentence in eval_data:
            file.write(sentence["text"] + "\n")

In [ ]:
import re

for split in ["dev", "test"]:
    tp = 0
    fp = 0
    fn = 0
    wt = 0
    with open(f"llama-{split}-true.txt", "r", encoding="utf-8") as file_true, \
         open(f"llama-{split}-pred.txt", "r", encoding="utf-8") as file_pred, \
         open(f"llama-{split}-false-positives.txt", "w", encoding="utf-8") as file_fp:
        for line_num, (true, pred) in enumerate(zip(file_true, file_pred), start=1):
            true_mentions = re.findall("<<(.*?)>>", true)
            true_mentions_dict = {}
            for mention in true_mentions:
                entity_type, mention = mention.split(": ", 1)
                true_mentions_dict[mention] = entity_type

            pred_mentions = re.findall("<<(.*?)>>", pred)
            pred_mentions_dict = {}
            for mention in pred_mentions:
                try:
                    entity_type, mention = mention.split(": ", 1)
                    pred_mentions_dict[mention] = entity_type
                except:
                    print("Invalid mention: " + mention)

            for mention, true_type in true_mentions_dict.items():
                pred_type = pred_mentions_dict.get(mention)
                if pred_type == true_type:
                    tp += 1
                elif pred_type is None:
                    fn += 1
                else:   # partial credit for wrong type
                    tp += 0.5
                    fn += 0.5
                    fp += 0.5
                    wt += 1
            for mention in pred_mentions_dict:
                if true_mentions_dict.get(mention) is None:
                    fp += 1
                    file_fp.write(f"{line_num} {mention}\n")

    print(f"***{split}***")
    print(tp, fp, fn, wt)
    print(f"Precision: {tp / (tp + fp)}")
    print(f"Recall: {tp / (tp + fn)}")

# Round 2

## Setup + Data Prep

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
# TODO: tweak prompt; train on completions only?

prompt = """You are an expert annotator tasked with labeling a word/phrase from a sentence.
If the word/phrase is a named entity, return its type (PER, LOC, ORG, or MISC).
If the word/phrase is not a named entity, return O.

### Input:
{}

### Response:
{}"""

from datasets import load_dataset
dev_split = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = "dev")
tag_int2str = dev_split.features["ner_tags"].feature.int2str

data_for_training = []

for tokens, tags in zip(dev_split["tokens"], dev_split["ner_tags"]):
    if tokens[0] == "RT":
        tokens = tokens[3:]
        tags = tags[3:]
    entities = []
    curr_mention = ""
    curr_tag = ""
    for token, tag_int in zip(tokens, tags):
        tag = tag_int2str(tag_int)
        if tag.startswith("I"):
            curr_mention += f"{token} "
        else:
            if curr_mention:
                entities.append((curr_mention, curr_tag))
                curr_mention = []
            if tag.startswith("B"):
                curr_mention = f"{token} "
                curr_tag = tag[2:]
    if curr_mention:
        entities.append((curr_mention, curr_tag))
    sentence = " ".join(tokens)
    for entity in entities:
        input = f'Sentence: "{sentence}". The type of "{entity[0]}" is:'
        output = entity[1]
        data_for_training.append(prompt.format(input, output) + tokenizer.eos_token)

In [ ]:
with open("llama-dev-false-positives.txt", "r", encoding="utf-8") as file_fp:
    for line in file_fp:
        line_num, mention = line.split(" ", 1)
        sentence_tokens = dev_split["tokens"][int(line_num)-1]
        if sentence_tokens[0] == "RT":
            sentence_tokens = sentence_tokens[3:]
        sentence = " ".join(sentence_tokens)
        input = f'Sentence: "{sentence}". The type of "{mention}" is:'
        data_for_training.append(prompt.format(input, "O") + tokenizer.eos_token)

In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_dict({"text": data_for_training})

## Training

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

## Eval

In [ ]:
import csv
test_split = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = "test")
tag_int2str = test_split.features["ner_tags"].feature.int2str
test_data = []

for tokens, tags in zip(test_split["tokens"], test_split["ner_tags"]):
    if tokens[0] == "RT":
        tokens = tokens[3:]
        tags = tags[3:]
    entities = []
    curr_mention = ""
    curr_tag = ""
    for token, tag_int in zip(tokens, tags):
        tag = tag_int2str(tag_int)
        if tag.startswith("I"):
            curr_mention += f"{token} "
        else:
            if curr_mention:
                entities.append((curr_mention, curr_tag))
                curr_mention = []
            if tag.startswith("B"):
                curr_mention = f"{token} "
                curr_tag = tag[2:]
    if curr_mention:
        entities.append((curr_mention, curr_tag))
    sentence = " ".join(tokens)
    for entity in entities:
        test_data.append((sentence, entity[0], entity[1]))

In [ ]:
with open("llama-test-false-positives.txt", "r", encoding="utf-8") as file_fp:
    for line in file_fp:
        line_num, mention = line.split(" ", 1)
        sentence_tokens = test_split["tokens"][int(line_num)-1]
        if sentence_tokens[0] == "RT":
            sentence_tokens = sentence_tokens[3:]
        sentence = " ".join(sentence_tokens)
        test_data.append((sentence, mention, "O"))

In [ ]:
import random
random.seed(0)
random.shuffle(test_data)

with open("llama-test-true.csv", "w", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Sentence", "Mention", "Type"])
    writer.writerows(test_data)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

with open("llama-test-pred-r2.csv", "w", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Sentence", "Mention", "Type"])
    for row in test_data:
        user_input = f'Sentence: "{row[0]}". The type of "{row[1]}" is:'
        input = tokenizer(
            [
                prompt.format(
                    user_input, # input
                    "", # output - leave this blank for generation!
                )
            ], return_tensors = "pt").to("cuda")
        output = model.generate(**input, use_cache = True)
        output = tokenizer.batch_decode(output[:, input.input_ids.shape[1]:], skip_special_tokens = True)
        print(output)
        writer.writerow([row[0], row[1], output[0]])

In [ ]:
import re
tp = 0
fp = 0
fn = 0
wt = 0
with open("llama-test-true.csv", "r", encoding="utf-8") as file_true, \
     open("llama-test-pred-r2.csv", "r", encoding="utf-8") as file_pred:
    reader_true = csv.reader(file_true)
    reader_pred = csv.reader(file_pred)
    next(reader_true)   # skip header row
    next(reader_pred)
    for true, pred in zip(reader_true, reader_pred):
        if true[2] == pred[2]:
            if true[2] != "O":
                tp += 1
        elif true[2] == "O":
            fp += 1
        elif pred[2] == "O":
            fn += 1
        else:   # partial credit for wrong type
            tp += 0.5
            fn += 0.5
            fp += 0.5
            wt += 1

print(tp, fp, fn, wt)
print(f"Precision: {tp / (tp + fp)}")
print(f"Recall: {tp / (tp + fn)}")